# Results Analysis and Comparison
Analyze and visualize metrics from experiment results CSV files

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

# Define results directory
results_dir = './results'
print(f"Results directory: {os.path.abspath(results_dir)}")

## Load CSV Files from Directory

In [ ]:
# Load all CSV files from results directory
csv_files = glob.glob(os.path.join(results_dir, '*.csv'))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

# Load all CSVs into a dictionary
data_frames = {}
for filepath in csv_files:
    filename = os.path.basename(filepath)
    try:
        df = pd.read_csv(filepath)
        data_frames[filename] = df
        print(f"✓ Loaded {filename}: {df.shape[0]} rows × {df.shape[1]} columns")
    except Exception as e:
        print(f"✗ Error loading {filename}: {e}")

print(f"\nSuccessfully loaded {len(data_frames)} files")

## Explore Data Structure

In [ ]:
# Display information about each dataset
for filename, df in data_frames.items():
    print(f"\n{'='*80}")
    print(f"File: {filename}")
    print(f"{'='*80}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns ({len(df.columns)}):")
    for i, col in enumerate(df.columns):
        print(f"  {i+1:2d}. {col}: {df[col].dtype}")
    
    print(f"\nFirst few rows:")
    display(df.head(2))
    
    print(f"\nBasic Statistics:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        display(df[numeric_cols].describe())

## Analyze NBR Base Estimators Results

In [ ]:
# Filter for nbr_base_estimators files (main experiment results)
nbr_files = [f for f in data_frames.keys() if 'nbr_base_estimators' in f]
print(f"Found {len(nbr_files)} nbr_base_estimators files:")
for f in nbr_files:
    print(f"  - {f}")

# Combine all nbr_base_estimators files for comparison
if nbr_files:
    dfs_to_combine = [data_frames[f] for f in nbr_files]
    combined_df = pd.concat(dfs_to_combine, ignore_index=True)
    print(f"\nCombined nbr_base_estimators data: {combined_df.shape}")
    print(f"Methods: {combined_df['method'].unique()}")
    print(f"Datasets: {combined_df['dataset'].unique()}")
    print(f"Black boxes: {combined_df['black_box'].unique()}")
    print(f"K values: {sorted(combined_df['k'].unique())}")
else:
    print("No nbr_base_estimators files found")

## Plot Metrics vs K (Multi-panel Comparison)

In [ ]:
# Define metrics to plot (similar to the paper image)
metrics_to_plot = [
    ('perc_valid_cf', 'Validity (%)'),
    ('distance_l2', 'Distance (L2)'),
    ('distance_mad', 'Distance (MAD)'),
    ('diversity_l2', 'Diversity (L2)'),
    ('avg_nbr_changes_per_cf', 'Sparsity (Avg Changes)'),
    ('runtime', 'Runtime (sec)')
]

# Filter for a specific dataset/model combination for cleaner visualization
if 'combined_df' in locals():
    # Get all unique dataset-model combinations
    combinations = combined_df[['dataset', 'black_box']].drop_duplicates()
    
    # Create plots for first combination
    for idx, row in combinations.head(1).iterrows():
        dataset = row['dataset']
        model = row['black_box']
        
        # Filter data for this combination
        plot_data = combined_df[(combined_df['dataset'] == dataset) & (combined_df['black_box'] == model)].copy()
        
        if len(plot_data) == 0:
            print(f"No data for {dataset} - {model}")
            continue
            
        print(f"\nPlotting metrics for: {dataset} - {model}")
        print(f"Methods: {plot_data['method'].unique()}")
        print(f"K values: {sorted(plot_data['k'].unique())}")
        
        # Create multi-panel plot
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        fig.suptitle(f'Metrics Comparison: {dataset} ({model})', fontsize=16, fontweight='bold')
        axes = axes.flatten()
        
        for idx, (metric, title) in enumerate(metrics_to_plot):
            ax = axes[idx]
            
            # Check if metric exists in data
            if metric not in plot_data.columns:
                ax.text(0.5, 0.5, f'Metric "{metric}" not found', 
                       ha='center', va='center', transform=ax.transAxes)
                ax.set_title(title)
                continue
            
            # Get unique methods
            methods = sorted(plot_data['method'].unique())
            
            # Plot each method
            for method in methods:
                method_data = plot_data[plot_data['method'] == method].sort_values('k')
                if len(method_data) > 0:
                    # Handle different metric types
                    if metric == 'perc_valid_cf':
                        # Convert to percentage if needed
                        y_vals = method_data[metric] * 100 if method_data[metric].max() <= 1 else method_data[metric]
                    else:
                        y_vals = method_data[metric]
                    
                    ax.plot(method_data['k'], y_vals, marker='o', label=method, linewidth=2, markersize=6)
            
            ax.set_xlabel('K', fontsize=11, fontweight='bold')
            ax.set_ylabel(title, fontsize=11)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend(fontsize=9, loc='best')
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n✓ Plot generated for {dataset} - {model}")

## Interactive Method Comparison

In [ ]:
# Create visualization for each dataset-model combination
if 'combined_df' in locals():
    combinations = combined_df[['dataset', 'black_box']].drop_duplicates()
    
    for combo_idx, combo_row in combinations.iterrows():
        dataset = combo_row['dataset']
        model = combo_row['black_box']
        
        plot_data = combined_df[(combined_df['dataset'] == dataset) & (combined_df['black_box'] == model)].copy()
        
        if len(plot_data) == 0:
            continue
        
        print(f"\n{'='*80}")
        print(f"Comparison: {dataset} - {model}")
        print(f"{'='*80}")
        
        # Summary statistics
        print(f"\nDataset Info:")
        print(f"  Methods: {', '.join(sorted(plot_data['method'].unique()))}")
        print(f"  K range: {plot_data['k'].min()} to {plot_data['k'].max()}")
        print(f"  Total rows: {len(plot_data)}")
        
        # Method comparison table
        method_stats = plot_data.groupby('method').agg({
            'perc_valid_cf': 'mean',
            'distance_l2': 'mean',
            'runtime': 'mean',
            'k': 'count'
        }).round(4)
        method_stats.columns = ['Avg Validity%', 'Avg Distance', 'Avg Runtime', 'Samples']
        
        print(f"\nMethod Performance Summary:")
        display(method_stats)

## All Available Metrics

In [ ]:
# Display all available numeric columns
if 'combined_df' in locals():
    numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Available numeric metrics ({len(numeric_cols)}):")
    for i, col in enumerate(numeric_cols, 1):
        print(f"  {i:2d}. {col}")
        
    # Show data types
    print(f"\nColumn Data Types:")
    for col in combined_df.columns:
        print(f"  {col}: {combined_df[col].dtype}")

## Heatmap Comparison

In [ ]:
# Create heatmaps showing method performance across k values
if 'combined_df' in locals():
    combinations = combined_df[['dataset', 'black_box']].drop_duplicates()
    
    for combo_idx, combo_row in combinations.iterrows():
        dataset = combo_row['dataset']
        model = combo_row['black_box']
        
        plot_data = combined_df[(combined_df['dataset'] == dataset) & (combined_df['black_box'] == model)].copy()
        
        if len(plot_data) == 0:
            continue
        
        # Create heatmap for average validity across methods and k
        pivot_data = plot_data.pivot_table(
            values='perc_valid_cf',
            index='method',
            columns='k',
            aggfunc='mean'
        )
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'{dataset} ({model}) - Method Performance', fontsize=14, fontweight='bold')
        
        # Heatmap 1: Validity
        sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[0], cbar_kws={'label': 'Validity'})
        axes[0].set_title('Average Validity by Method and K')
        axes[0].set_ylabel('Method')
        axes[0].set_xlabel('K')
        
        # Heatmap 2: Distance
        pivot_distance = plot_data.pivot_table(
            values='distance_l2',
            index='method',
            columns='k',
            aggfunc='mean'
        )
        
        sns.heatmap(pivot_distance, annot=True, fmt='.3f', cmap='YlGnBu_r', ax=axes[1], cbar_kws={'label': 'Distance'})
        axes[1].set_title('Average Distance (L2) by Method and K')
        axes[1].set_ylabel('Method')
        axes[1].set_xlabel('K')
        
        plt.tight_layout()
        plt.show()

## Comparison Summary Report

In [ ]:
print("\n" + "="*80)
print("COMPREHENSIVE RESULTS COMPARISON SUMMARY")
print("="*80)

# Summary of all files
print(f"\n📊 Data Files Summary:")
print(f"  Total files: {len(data_frames)}")
for filename, df in data_frames.items():
    print(f"    - {filename}: {df.shape[0]} rows × {df.shape[1]} columns")

if 'combined_df' in locals():
    print(f"\n📈 Combined NBR Base Estimators Data:")
    print(f"  Total rows: {len(combined_df)}")
    print(f"  Datasets: {combined_df['dataset'].nunique()} unique ({', '.join(combined_df['dataset'].unique())})")
    print(f"  Methods: {combined_df['method'].nunique()} unique ({', '.join(sorted(combined_df['method'].unique()))})")
    print(f"  Black Boxes: {combined_df['black_box'].nunique()} unique ({', '.join(combined_df['black_box'].unique())})")
    print(f"  K range: {int(combined_df['k'].min())} to {int(combined_df['k'].max())}")
    
    # Find best method by various metrics
    print(f"\n🏆 Best Performing Methods:")
    
    # Best validity
    best_validity = combined_df.loc[combined_df['perc_valid_cf'].idxmax()]
    print(f"  Best Validity: {best_validity['method']} ({best_validity['perc_valid_cf']*100:.1f}%)")
    
    # Best (lowest) distance
    best_distance = combined_df.loc[combined_df['distance_l2'].idxmin()]
    print(f"  Best Distance: {best_distance['method']} ({best_distance['distance_l2']:.4f})")
    
    # Best (lowest) runtime
    best_runtime = combined_df.loc[combined_df['runtime'].idxmin()]
    print(f"  Fastest: {best_runtime['method']} ({best_runtime['runtime']:.4f} sec)")

print("\n✓ Analysis complete!")